In [1]:
pip install transformers scikit-learn torch numpy pandas peft ollama triton bitsandbytes

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#CLEAR DISK SPACE AND CACHE

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
torch.cuda.empty_cache()
#torch.cuda.reset_peak_memory_stats()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [3]:
#IMPORT PACKAGES AND MODEL

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch

model_name = "openai/gpt-oss-20b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name, 
                                             device_map = "auto",
                                             )

pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype="auto",
    device_map = "auto"
)



2025-08-14 17:31:38.975354: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-14 17:31:38.987865: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755207099.002717  498732 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755207099.007379  498732 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755207099.018947  498732 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


In [ ]:

#LOAD DATA AND PACKAGES

import pandas as pd
import numpy as np
import sklearn as sk
import json
import ollama
from pathlib import Path

#unlabelled dataset
df = pd.read_csv("../unlabelled_relevance.csv")

#labelled_dataset from Annotator 1 in round 2
df_gold = pd.read_csv('../Human/A1_r2.csv')

with open('../relevance_codebook.json', 'r') as f:
    codebook = json.load(f)
codebook_str = json.dumps(codebook, indent=2, ensure_ascii=False)


In [ ]:
#helper functions

##parsing responses from model
def parse_json_with_fallback(content):
    try:
        dict_val = json.loads(content)
        str_val = dict_val.get('Relevance')
        binary_val = 1 if str_val.strip().lower() == "yes" else 0

        return binary_val
    except json.JSONDecodeError:
        return pd.NA


##compute kappa and accuracy
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]

    #join these two dataframes
    df_temp = (
        df_pred_reduced
            .merge(df_gold_reduced, on=column_match, how='inner')   # keep only matching IDs
            .dropna(subset=[column_pred, column_gold])          # drop rows where values are NaN
            .reset_index(drop=True)                                 # tidy up the index
    )
    
    #ensure that the columns have data in the same type 
    df_temp[column_pred] = df_temp[column_pred].astype(int)
    df_temp[column_gold] = df_temp[column_gold].astype(int)

    #compute scores
    acc = round(100*sk.metrics.accuracy_score(df_temp[column_gold], df_temp[column_pred]), 3)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold], df_temp[column_pred]), 3)

    print("Accuracy is " + str(acc) + "%, and Cohen's kappa is " + str(k))



In [ ]:
#ZERO SHOT: WITH CODEBOOK

#Prompts
SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform relevance coding on news articles, by classifying articles as relevant, or not relevant. 
Remember to prioritize accuracy and clarity in your analysis, using the provided context and your expertise to guide your evaluation
Here's the codebook for you to follow for class definitions: """ + codebook_str


USER_PROMPT =  """Is the following article relevant?  Provide your response in a JSON dict format, as follows, and include nothing else in the response : { "Relevance": "yes/no" }. 
If you are uncertain about the classification, force a decision to choose "yes" or "no". """

#sample for testing things first
#df_test = df.sample(n= 5, random_state= 42).reset_index()

df_test = df

df1 = df_test

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = pipeline(
        messages,
        max_new_tokens=2048)

        predictions.append(outputs[0]["generated_text"][-1])
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")

#save responses
with open("Predicted/gpt-oss-20b-zshot_predictions.txt", "w") as file:
    for item in predictions:
        file.write(f"{item}\n")


0 iterations finished


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


10 iterations finished
20 iterations finished
30 iterations finished
40 iterations finished
50 iterations finished
60 iterations finished
70 iterations finished
80 iterations finished
90 iterations finished
100 iterations finished
110 iterations finished
120 iterations finished
130 iterations finished
140 iterations finished
150 iterations finished
160 iterations finished
170 iterations finished
180 iterations finished
190 iterations finished
200 iterations finished
210 iterations finished
220 iterations finished
230 iterations finished
240 iterations finished
250 iterations finished
260 iterations finished
270 iterations finished
280 iterations finished
290 iterations finished
300 iterations finished
310 iterations finished
320 iterations finished
330 iterations finished
340 iterations finished
350 iterations finished
360 iterations finished
370 iterations finished
380 iterations finished
390 iterations finished
400 iterations finished
410 iterations finished
420 iterations finished
4

In [ ]:
#PARSE GPT RESPONSES AND CONVERT TO WHAT WE NEED

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch

#load the previous output
with open("Predicted/gpt-oss-20b-zshot_predictions.txt", 'r') as file:
    predictions = file.readlines()
    
model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

#8-bit quantise
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,               
    llm_int8_threshold=6.0,           
    llm_int8_enable_fp32_cpu_offload=True,
    llm_int8_skip_modules=["output_layer"]
)

model = AutoModelForCausalLM.from_pretrained(model_name, 
                                             quantization_config=bnb_config, #if quantizing
                                             device_map = "auto",
                                             #cache_dir = "/tmp"
                                             )

pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

#PROMPTS
SYSTEM_PROMPT = """You're a simple NLP tool that parses another models response and outputs in a specified format.
An LLM performed a content analytic task and output a string as a response with answers and justifications. 
Your job is to parse the response and just output the answers without justification. 

Provide your response in a JSON format, as follows, and include NOTHING ELSE in the response: 
{"Relevance": "yes/no"}
"""

USER_PROMPT =  """Note: Sometimes the answer may be incomplete from the LLM's end. In that case, just output NA. 
Parse the response and output the answers in correct format for the following LLM response: """

answers = []

#now parse through the output
for i in range(len(predictions)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + predictions[i] 
        }

            ]
    
        outputs = pipeline(
        messages,
        max_new_tokens=2048)

        answers.append(outputs[0]["generated_text"][-1])
    
    
    #if there is an error
    except Exception as e:
            answers.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")

#save the responses
RelevanceList = []

for output in answers:
    if output == None:
        RelevanceList.append(None)

    else:
        content = output.get('content')
        content = content.replace("[", "").replace("]", "")

        try: 
            annotation = parse_json_with_fallback(content)
            RelevanceList.append(annotation)

        #or maybe not
        except Exception as e:
            RelevanceList.append(None)

df1["relevance"] = RelevanceList

#compute metrics
compute_scores(df_gold= df_gold, df_pred = df1, column_match= 'stories_id', column_gold= 'gold', column_pred= 'relevance')
    
df1.to_csv('Predicted/gpt-oss-20b-zshot_withCB.csv')

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


0 iterations finished


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


10 iterations finished


...

480 iterations finished


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


490 iterations finished


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Accuracy is 91.583%, and Cohen's kappa is 0.8
